# z012 - Optuna con rango más amplio de min_sum_hessian_in_leaf

En z005 el mejor (trial 34) tuvo hessian 9,76, **pegado al tope del rango (10)**: el óptimo puede estar más arriba.
Ahora: hessian de 1 a 200 y num_leaves de 16 a 512 (escala logarítmica), 40 trials, 1 semilla.
Después se **revalidan los 5 mejores con 5 semillas** junto con el trial 34 como referencia.

Estudio nuevo `z012_rango_s501419` en `C:/Users/keila/optuna/z012.db` (fuera de OneDrive; si se corta, retoma).

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Optuna ---
import optuna
train = df[df["foto_mes"] == 202104]
datos_train = lgb.Dataset(train[variables], label=train["clase_binaria"], free_raw_data=False)
datos_valid = lgb.Dataset(valid[variables], label=valid["clase_binaria"], reference=datos_train, free_raw_data=False)
fijos = {k: v for k, v in params.items() if k not in ("num_leaves", "min_sum_hessian_in_leaf")}

def objetivo(trial):
    p = {**fijos, "metric": "auc", "seed": SEMILLAS[0],
         "num_leaves": trial.suggest_int("num_leaves", 16, 512, log=True),
         "min_sum_hessian_in_leaf": trial.suggest_float("min_sum_hessian_in_leaf", 1, 200, log=True)}
    modelo = lgb.train(p, datos_train, num_boost_round=3000, valid_sets=[datos_valid],
                       callbacks=[lgb.early_stopping(200, verbose=False)])
    best_iter = modelo.best_iteration or modelo.num_trees()
    trial.set_user_attr("best_iter", best_iter)
    return metricas(modelo.predict(valid[variables], num_iteration=best_iter))[0]

optuna.logging.set_verbosity(optuna.logging.WARNING)
estudio = optuna.create_study(study_name="z012_rango_s501419", storage="sqlite:///C:/Users/keila/optuna/z012.db",
                              load_if_exists=True, direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=SEMILLAS[0]))
hechos = len([t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE])
estudio.optimize(objetivo, n_trials=max(0, 40 - hechos))

trials = estudio.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials = trials[trials["state"] == "COMPLETE"].sort_values("value", ascending=False)
trials.to_csv("z012_optuna_trials.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
trials.head(10)

,number,value,params_min_sum_hessian_in_leaf,params_num_leaves,user_attrs_best_iter,state
26,26,440.8800,115.008759,29,2929,COMPLETE
32,32,438.7075,107.120969,47,3000,COMPLETE
5,5,437.0300,115.510680,16,2949,COMPLETE
13,13,435.5175,97.983251,20,2748,COMPLETE
33,33,434.6375,119.312552,483,3000,COMPLETE
8,8,434.2250,169.782525,20,2997,COMPLETE
11,11,434.1150,167.278645,49,3000,COMPLETE
28,28,434.0600,85.157672,66,2419,COMPLETE
35,35,433.4000,110.907428,44,2984,COMPLETE
31,31,433.1525,125.242280,26,2993,COMPLETE


In [5]:
# --- 5. Revalidar los 5 mejores (y el trial 34 de z005) con 5 semillas ---
candidatos = {"z005 trial 34 (referencia)": (137, 9.755574, 538)}
for _, t in trials.head(5).iterrows():
    candidatos[f"z012 trial {int(t['number'])}"] = (int(t["params_num_leaves"]),
                                                     float(t["params_min_sum_hessian_in_leaf"]),
                                                     int(t["user_attrs_best_iter"]))
probs = {}
for nombre, (hojas, hessian, arboles) in candidatos.items():
    p = {**fijos, "num_leaves": hojas, "min_sum_hessian_in_leaf": hessian}
    probs[f"{nombre}: hojas {hojas}, hessian {hessian:.2f}, {arboles} árboles"] = [
        lgb.train({**p, "seed": s}, datos_train, num_boost_round=arboles).predict(valid[variables]) for s in SEMILLAS]
    print(nombre, "revalidado")

z005 trial 34 (referencia) revalidado


z012 trial 26 revalidado


z012 trial 32 revalidado


z012 trial 5 revalidado


z012 trial 13 revalidado


z012 trial 33 revalidado


In [6]:
# --- 6. Comparación ---
por_semilla, tabla = resumen(probs)
tabla = tabla.sort_values(("ganancia_max_M", "mean"), ascending=False)
tabla.to_csv("z012_revalidacion.csv", sep=";", decimal=",", encoding="utf-8-sig")
tabla

ganancia_max_M       \
                                                             mean  std   
version                                                                  
z012 trial 26: hojas 29, hessian 115.01, 2929 á...          438.2  3.1   
z012 trial 32: hojas 47, hessian 107.12, 3000 á...          437.1  2.9   
z012 trial 5: hojas 16, hessian 115.51, 2949 ár...          435.5  2.4   
z012 trial 33: hojas 483, hessian 119.31, 3000 ...          435.0  2.8   
z012 trial 13: hojas 20, hessian 97.98, 2748 ár...          434.7  1.9   
z005 trial 34 (referencia): hojas 137, hessian ...          399.6  2.3   

                                                   top12000_M       mejor_n  \
                                                         mean  std     mean   
version                                                                       
z012 trial 26: hojas 29, hessian 115.01, 2929 á...      433.0  4.7  12331.6   
z012 trial 32: hojas 47, hessian 107.12, 3000 á...      431.2  2.5  12427.4   
z012 trial 5: hojas 16, hessian 115.51, 2949 ár...      432.7  3.4  12132.6   
z012 trial 33: hojas 483, hessian 119.31, 3000 ...      430.1  4.7  12029.8   
z012 trial 13: hojas 20, hessian 97.98, 2748 ár...      431.9  3.5  12262.8   
z005 trial 34 (referencia): hojas 137, hessian ...      392.9  3.7  13212.0   

                                                           shuffle_split  \
                                                       std          mean   
version                                                                    
z012 trial 26: hojas 29, hessian 115.01, 2929 á...   132.4         216.5   
z012 trial 32: hojas 47, hessian 107.12, 3000 á...   102.5         217.0   
z012 trial 5: hojas 16, hessian 115.51, 2949 ár...   380.9         216.6   
z012 trial 33: hojas 483, hessian 119.31, 3000 ...   300.1         215.9   
z012 trial 13: hojas 20, hessian 97.98, 2748 ár...   598.5         215.6   
z005 trial 34 (referencia): hojas 137, hessian ...  1072.1         199.1   

                                                                      \
                                                     std veces_mejor   
version                                                                
z012 trial 26: hojas 29, hessian 115.01, 2929 á...  11.6          13   
z012 trial 32: hojas 47, hessian 107.12, 3000 á...  11.5          16   
z012 trial 5: hojas 16, hessian 115.51, 2949 ár...  11.5          10   
z012 trial 33: hojas 483, hessian 119.31, 3000 ...  11.5           4   
z012 trial 13: hojas 20, hessian 97.98, 2748 ár...  11.7           7   
z005 trial 34 (referencia): hojas 137, hessian ...  11.6           0   

                                                       ensamble_5  
                                                   ganancia_max_M  
version                                                            
z012 trial 26: hojas 29, hessian 115.01, 2929 á...          438.1  
z012 trial 32: hojas 47, hessian 107.12, 3000 á...          437.8  
z012 trial 5: hojas 16, hessian 115.51, 2949 ár...          438.9  
z012 trial 33: hojas 483, hessian 119.31, 3000 ...          436.0  
z012 trial 13: hojas 20, hessian 97.98, 2748 ár...          434.2  
z005 trial 34 (referencia): hojas 137, hessian ...          402.4

In [7]:
# --- 7. Anotar ---
anotar(tabla, "z012 optuna rango", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
- **Gran mejora**: los 5 mejores dan **434,7 a 438,2 M** vs 399,6 del trial 34 → **+35 a +38 M**, con desvío bajo (2-3 M).
- Zona buena: **hessian ~100-120** (muy por encima del tope 10 de z005), **pocas hojas (16-50)**.
- Ojo: usan **~2.750-3.000 árboles = el tope (3000)** → probablemente rinda más con un tope mayor.
- Mejor N ~12.000-12.400 (coincide con lo que veníamos mandando).
- Decisión: pendiente (candidato fuerte para rehacer el ensamble final).